# v2 승격 결정 노트북 (A안) — 운영점 비교로 최종 판단

> 계획: `docs/final_model_leakfree_switch_plan.md` §9-4 승격 기준.
> 흐름: **sweepA 학습 → 운영점(생존율 동기화) F1 vs exp47 → 누수 요약 → 승격 판정 → (승격 시) artifact export**

```
승격 기준 (§9-4):
  (a) test PR-AUC > exp47(0.570)          ← 랭킹 품질
  (b) 운영점 F1 ≥ exp47(0.666)            ← MD 합격선 실제 성능  ★이 노트북의 핵심
  (c) leak-free                            ← train-only basket Δ≈0
  (d) gap < 0.10 (참고)                    ← 과적합
→ (a)&(b)&(c) 충족 시 승격 권장.
```

In [1]:
import os, sys, json
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.chdir(ROOT)
import numpy as np, pandas as pd
from experiments.v2_multitask import run, CFG
from experiments.v2_promote import operating_point, compare_models, export_v2, decision
from experiments.v2_sweep_and_leakcheck import structural_leak_check
SWEEPA = dict(dropout=0.5, dropedge=0.35, hidden_dim=32, aux_lambda=1.0)
print("sweepA cfg:", SWEEPA)

C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sweepA cfg: {'dropout': 0.5, 'dropedge': 0.35, 'hidden_dim': 32, 'aux_lambda': 1.0}


## 1. v2-sweepA 학습 (leak-free 멀티태스크 + 정규화)

In [2]:
model, metrics, ctx = run(full=True, cfg_override=SWEEPA, tag="sweepA")
print("train/val/test PR-AUC:", {k: round(metrics[k]['pr_auc'],4) for k in ['train','val','test']}, "| gap:", round(metrics['gap'],3))

v2 graph: +basket_comp 엣지 8,846 | aux 동반구매 양성쌍(train) 689 | hidden=32


[001] main=1.0884 aux=2.5504 val_pr=0.2975


[005] main=1.0335 aux=1.1921 val_pr=0.3656


[010] main=1.0138 aux=0.7465 val_pr=0.3874


[015] main=1.0030 aux=0.5798 val_pr=0.4128


...

[130] main=0.6704 aux=0.0853 val_pr=0.5895


[135] main=0.6678 aux=0.1064 val_pr=0.5876


[140] main=0.6357 aux=0.0855 val_pr=0.5920


[145] main=0.6459 aux=0.0872 val_pr=0.5944


[150] main=0.6401 aux=0.0980 val_pr=0.5945
early stop @ 150
────────────────────────────────────────────────────────────
[sweepA] train_pr=0.7334 val_pr=0.5980 test_pr=0.6051 test_auc=0.8313 test_f1=0.6127 | gap=0.135
  cfg: dropout=0.5 dropedge=0.35 hidden=32 wd=0.002 aux_λ=1.0
  vs exp47: test 0.5699 / gap 0.215 | vs v2-base: test 0.6003 / gap 0.190
train/val/test PR-AUC: {'train': 0.7334, 'val': 0.598, 'test': 0.6051} | gap: 0.135


## 2. ★ 운영점 비교 (생존율 23.8% 동기화) — v2 vs exp47

In [3]:
cmp_df, op_v2, op_47 = compare_models(ctx['prob_full'], ctx['y'])
display(cmp_df[['thr','predpos','precision','recall','f1','TP','FP','FN']])
print(f"운영점 F1:  exp47 {op_47['f1']}  vs  v2-sweepA {op_v2['f1']}   (Δ {op_v2['f1']-op_47['f1']:+.4f})")
print(f"test PR-AUC: exp47 0.5699  vs  v2-sweepA {metrics['test']['pr_auc']:.4f}   (Δ {metrics['test']['pr_auc']-0.5699:+.4f})")

,thr,predpos,precision,recall,f1,TP,FP,FN
exp47,0.7755,0.2378,0.6658,0.6658,0.6658,797.0,400.0,400.0
v2_sweepA,0.7022,0.2378,0.6349,0.6349,0.6349,760.0,437.0,437.0


운영점 F1:  exp47 0.6658  vs  v2-sweepA 0.6349   (Δ -0.0309)
test PR-AUC: exp47 0.5699  vs  v2-sweepA 0.6051   (Δ +0.0352)


## 3. 누수 재검증 요약 (§9-3)

In [4]:
leak = structural_leak_check()
# 행동 검증(train-only basket)은 v2_sweep_and_leakcheck.py에서 실측: test 0.5921 ≈ full 0.6003 (Δ0.008)
leak_free = leak['aux_train_only'] and (leak['jaccard'] >= 0.5)   # 구조 + (사전 행동검증 Δ0.008)
print(f"aux 100% train: {leak['aux_train_only']} | basket Jaccard: {leak['jaccard']} | 행동검증(사전): train-only 0.592 ≈ full 0.600")
print("→ leak-free:", leak_free)

[A] 구조 누수 검증 (§9-3)


  aux 양성쌍 689개 — 전부 train? True  (False면 누수)


  basket_comp(support≥3): full 8846 / train-only 5231 / Jaccard 0.591
  → Jaccard 높으면 구조가 test 제품에 거의 무의존(누수 희석). test-전용 엣지 3615개
aux 100% train: True | basket Jaccard: 0.591 | 행동검증(사전): train-only 0.592 ≈ full 0.600
→ leak-free: True


## 4. 승격 판정

In [5]:
dec = decision(metrics, op_v2, op_47, leak_free=leak_free)
print(json.dumps(dec, ensure_ascii=False, indent=2))
print()
print("="*50)
print(f"  판정: {dec['verdict']}")
print(f"  (a) PR-AUC 초과: {dec['pr_win']} | (b) 운영점 F1 동등이상: {dec['f1_ok']} | (c) leak-free: {dec['leak_free']} | (d) gap<0.10: {dec['gap_ok']}({dec['gap']})")
print("="*50)

{
  "promote": false,
  "pr_win": true,
  "f1_ok": false,
  "leak_free": true,
  "gap_ok": false,
  "gap": 0.135,
  "verdict": "⏸ 보류"
}

  판정: ⏸ 보류
  (a) PR-AUC 초과: True | (b) 운영점 F1 동등이상: False | (c) leak-free: True | (d) gap<0.10: False(0.135)


## 5. (승격 시) artifact export + 서빙 전환 안내

In [6]:
OUT = "experiments/results/v2_sweepA"
if dec['promote']:
    export_v2(model, ctx, metrics, OUT, op_v2)
    print("✅ export 완료 →", OUT)
    for fn in sorted(os.listdir(OUT)): print("   ", fn)
    print("\n[서빙 전환 절차]")
    print(" 1) src/eval/md/engine.py 및 serve.py 가 HINGNNv2+basket_comp 그래프를 로드하도록 어댑터 필요")
    print("    (engine._rebuild → HINGNNv2, build_graph에 keyword_basket_comp_edges 주입)")
    print(" 2) SERVING_EXP = 'v2_sweepA' / EngineConfig.exp_dir 변경")
    print(" 3) md_prescription_pipeline.ipynb 재실행 → 처방 캐시 재생성")
else:
    print("⏸ 보류 — exp47 유지. 기준 미달 항목:",
          [k for k in ['pr_win','f1_ok','leak_free'] if not dec[k]])

⏸ 보류 — exp47 유지. 기준 미달 항목: ['f1_ok']
